In [1]:
import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import writing_tools as wt
import utils

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False

RESULTS = {}

In [2]:
# Output the data for regressions in R

qual_df = dt.get_post_quality_analysis_data()
qual_df.to_parquet(os.path.join(DATA_PATH, "post_quality_analysis_data.parquet"))

quant_df = dt.get_post_quantity_analysis_data()
quant_df.to_parquet(os.path.join(DATA_PATH, "post_quantity_analysis_data.parquet"))

c:\Users\edwar\projects\sn-research\src\notebooks\../python\data_tools.py:91: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time
c:\Users\edwar\projects\sn-research\src\notebooks\../python\data_tools.py:91: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time
c:\Users\edwar\projects\sn-research\src\notebooks\../python\data_tools.py:91: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time
c:\Users\edwar\projects\sn-research\src\notebooks\../python\data_tools.py:91: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time


In [3]:
RESULTS = {
    'AvgPostingCost': f"{qual_df['cost'].mean():,.0f}",
    'AvgPostingCostCents': f"{qual_df['cost'].mean()/10:,.0f}",
    "AvgSats48": f"{qual_df['sats48'].mean():,.0f}",
    "AvgComments48": f"{qual_df['comments48'].mean():,.1f}",
    'SdPostingCost': f"{qual_df['cost'].std():,.0f}",
    "SdSats48": f"{qual_df['sats48'].std():,.0f}",
    "SdComments48": f"{qual_df['comments48'].std():,.1f}",
}
_ = wt.update_results(RESULTS)
RESULTS

{'AvgPostingCost': '51',
 'AvgPostingCostCents': '5',
 'AvgSats48': '364',
 'AvgComments48': '3.3',
 'SdPostingCost': '241',
 'SdSats48': '6,662',
 'SdComments48': '9.6'}

In [4]:
res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/03_pay_to_post_analysis.R"], capture_output=True, text=True, check=True)
print(res.stdout)

                            rzaps0             rzaps1             rzaps2           rzaps3
Dependent Var.:         log_sats48         log_sats48         log_sats48       log_sats48
                                                                                         
Constant         2.530*** (0.0760)                                                       
log_cost        0.3930*** (0.0480) 0.2728*** (0.0576) 0.2191*** (0.0498) 0.0915* (0.0436)
Fixed-Effects:  ------------------ ------------------ ------------------ ----------------
subId                           No                Yes                 No               No
weekId                          No                Yes                Yes              Yes
sub_subOwner_id                 No                 No                Yes              Yes
userId                          No                 No                 No              Yes
_______________ __________________ __________________ __________________ ________________
S.E.: Clus

In [5]:
# Extract quality and posting cost estimates

coefs_df = pd.read_parquet(
    os.path.join(DATA_PATH, "pay_to_post_quality_analysis_coefs.parquet")
)

idx = (coefs_df["regression_name"]=="rzaps2") & (coefs_df["coef_name"]=="log_cost")
coef = coefs_df.loc[idx, "estimate"].values[0]
elast = fr"{coef:.3f}"
double_effect = np.exp(coef*np.log(2)) - 1
double_effect_pct = fr"{np.abs(double_effect)*100:.1f}"
RESULTS = {
    'Zaps48CostElasticity': elast,
    'Zaps48CostDoublingEffect': double_effect_pct
}
_ = wt.update_results(RESULTS)
RESULTS


{'Zaps48CostElasticity': '0.219', 'Zaps48CostDoublingEffect': '16.4'}

In [6]:
# Effect of posting cost on post quality table

header = r"""\begin{table}[H]
\centering
\caption{Effect of Posting Cost on Post Quality} 
\vspace{0.2cm}
\label{tbl_sats48_cost_reg}
\begin{threeparttable}
\begin{tabular}{lcccc} 
\toprule
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
 & \multicolumn{4}{c}{ln(Zaps)} \\ 
 & (1) & (2) & (3) & (4)\\ 
\midrule
 &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular} 
\begin{tablenotes}[flushleft]
\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} Regression of number of sats a post earns in the first 48 hours on posting cost. The unit of observation is a post. Standard errors are clustered by territory.
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
reg_names = ["rzaps0", "rzaps1", "rzaps2", "rzaps3"]

vars = [
    ("ln(Posting Cost)", "log_cost"),
    ("Constant", "(Intercept)"),
]

tbl = ""
for v in vars:
    tbl += v[0] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[1])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[1])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1ex] " + "\n"

tbl += r""" & & & & \\
Territory FE            & N & Y & Y & Y \\
Week FE                 & N & Y & Y & Y \\
Territory Operator FE   & N & N & Y & Y \\
User FE                 & N & N & N & Y \\
& & & & \\
"""

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\" + "\n"

tbl += "R$^2$ "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="R2")
    r2 = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {r2:,.3f}"
tbl += r" \\ [1.8ex] " + "\n"



table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "results", "tbl_sats48_cost_reg.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)

\begin{table}[H]
\centering
\caption{Effect of Posting Cost on Post Quality} 
\vspace{0.2cm}
\label{tbl_sats48_cost_reg}
\begin{threeparttable}
\begin{tabular}{lcccc} 
\toprule
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
 & \multicolumn{4}{c}{ln(Zaps)} \\ 
 & (1) & (2) & (3) & (4)\\ 
\midrule
 &  &  &  &  \\
ln(Posting Cost)  & 0.393$^{***}$ & 0.273$^{***}$ & 0.219$^{***}$ & 0.091$^{**}$ \\
 & (0.048) & (0.058) & (0.050) & (0.044) \\ [1ex] 
Constant  & 2.530$^{***}$ &  &  &  \\
 & (0.076) &  &  &  \\ [1ex] 
 & & & & \\
Territory FE            & N & Y & Y & Y \\
Week FE                 & N & Y & Y & Y \\
Territory Operator FE   & N & N & Y & Y \\
User FE                 & N & N & N & Y \\
& & & & \\
Observations  & 179,480 & 179,477 & 179,476 & 178,197 \\
R$^2$  & 0.079 & 0.181 & 0.184 & 0.402 \\ [1.8ex] 
\bottomrule
\end{tabular} 
\begin{tablenotes}[flushleft]
\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:}

In [7]:
# Extract quantity and posting cost estimates

coefs_df = pd.read_parquet(
    os.path.join(DATA_PATH, "pay_to_post_quantity_analysis_coefs.parquet")
)

idx = (coefs_df["regression_name"]=="rquan2") & (coefs_df["coef_name"]=="log_cost")
coef = coefs_df.loc[idx, "estimate"].values[0]
elast = fr"{coef:.3f}"
double_effect = np.exp(coef*np.log(2)) - 1
double_effect_pct = fr"{np.abs(double_effect)*100:.1f}"

RESULTS = {
    'PostsCostElasticity': elast,
    'PostsCostDoublingEffect': double_effect_pct
}
_ = wt.update_results(RESULTS)
RESULTS

{'PostsCostElasticity': '-0.265', 'PostsCostDoublingEffect': '16.8'}

In [8]:
# Effect of posting cost on post quantity table

header = r"""\begin{table}[H]
\centering
\caption{Effect of Posting Cost on Post Quantity}
\vspace{0.2cm}
\label{tbl_posts_cost_reg}
\begin{threeparttable}
\begin{tabular}{lccc} 
\toprule
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
 & \multicolumn{3}{c}{ln(Posts)} \\ 
 & (1) & (2) & (3)\\ 
\midrule
 &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular} 
\begin{tablenotes}[flushleft]\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} Regression of number of posts in a territory on posting costs for that week. The unit of observation is a territory-week. Standard errors are clustered by territory.
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
reg_names = ["rquan0", "rquan1", "rquan2"]

vars = [
    ("log(Posting Cost)", "log_cost"),
    ("Constant", "(Intercept)"),
]

tbl = ""
for v in vars:
    tbl += v[0] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[1])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[1])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1ex]" + "\n"

tbl += r"""& & & \\
Territory FE            & N & Y & Y \\
Week FE                 & N & Y & Y \\
Territory Operator FE   & N & N & Y \\
& & & \\
"""

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ " + "\n"

tbl += r"R$^2$ "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="R2")
    r2 = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {r2:,.3f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "results", "tbl_posts_cost_reg.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)

\begin{table}[H]
\centering
\caption{Effect of Posting Cost on Post Quantity}
\vspace{0.2cm}
\label{tbl_posts_cost_reg}
\begin{threeparttable}
\begin{tabular}{lccc} 
\toprule
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
 & \multicolumn{3}{c}{ln(Posts)} \\ 
 & (1) & (2) & (3)\\ 
\midrule
 &  &  &  \\
log(Posting Cost)  & -0.242$^{***}$ & -0.266$^{***}$ & -0.265$^{***}$ \\
 & (0.079) & (0.063) & (0.074) \\ [1ex]
Constant  & 3.328$^{***}$ &  &  \\
 & (0.415) &  &  \\ [1ex]
& & & \\
Territory FE            & N & Y & Y \\
Week FE                 & N & Y & Y \\
Territory Operator FE   & N & N & Y \\
& & & \\
Observations  & 6,023 & 6,020 & 6,020 \\ 
R$^2$  & 0.092 & 0.776 & 0.788 \\ [1.8ex]
\bottomrule
\end{tabular} 
\begin{tablenotes}[flushleft]\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} Regression of number of posts in a territory on posting costs for that week. The unit of observation is a territory-week. S